# TP DES & AES — Partie 1 & 2 : chiffrement de Feistel personnalisé

## 1. Conception

On conçoit un chiffrement par blocs basé sur un **réseau de Feistel**. Le principe : on coupe le bloc en deux moitiés $L$ et $R$, et à chaque tour :

$$L_{i+1} = R_i \qquad R_{i+1} = L_i \oplus F(R_i, K_i)$$

L'intérêt de cette structure est que le déchiffrement est le même algorithme avec les sous-clés dans l'ordre inverse, **même si $F$ n'est pas inversible**.

### Choix des paramètres

| Paramètre | Choix | Justification |
|---|---|---|
| Taille de bloc | 64 bits (2 × 32 bits) | Même taille que DES → comparaison facile ; opérations sur des mots de 32 bits simples en Python |
| Taille de clé | 128 bits | 56 bits (DES) est cassable par force brute ; $2^{128}$ est hors de portée |
| Nombre de tours | 16 | L'effet avalanche est atteint dès ~4 tours (voir plus bas), 16 tours laissent une marge de sécurité, comme DES |
| Mode | ECB + padding PKCS#7 | Simple pour le TP (on discute la limite de l'ECB plus bas) |

### Fonction de tour $F(R, K)$

1. **Addition modulaire** : $x = (R + K) \bmod 2^{32}$. Contrairement au XOR, l'addition propage des retenues entre les bits → non-linéaire vis-à-vis du XOR.
2. **Couche de S-box** : on applique une S-box 4 bits → 4 bits (celle du chiffrement PRESENT) sur les 8 nibbles. C'est la source principale de **confusion** (relation non linéaire entre clé et chiffré).
3. **Couche linéaire de diffusion** : $y \oplus \text{rotl}(y,5) \oplus \text{rotl}(y,11)$. Chaque bit de sortie d'une S-box influence 3 positions différentes, donc plusieurs S-box au tour suivant → **diffusion**.

### Cadencement de clé

La clé de 128 bits est découpée en 4 mots $W_0..W_3$ de 32 bits. On garde un état $k$ :

$$k_{i} = \text{rotl}\big((k_{i-1} + W_{i \bmod 4} + C\cdot(i+1)) \bmod 2^{32},\ 7\big)$$

avec $C = \texttt{0x9E3779B9}$ (constante du nombre d'or, utilisée dans TEA). La constante dépendant du tour évite que deux tours aient la même sous-clé (attaques par glissement), et l'addition + rotation mélange les mots de la clé.

In [1]:
import os
import random

M32 = 0xFFFFFFFF
SBOX = [0xC, 0x5, 0x6, 0xB, 0x9, 0x0, 0xA, 0xD, 0x3, 0xE, 0xF, 0x8, 0x4, 0x7, 0x1, 0x2]  # S-box PRESENT
ROUNDS = 16
C = 0x9E3779B9

def rotl(x, s):
    return ((x << s) | (x >> (32 - s))) & M32

def F(R, K):
    x = (R + K) & M32                     # addition modulaire
    y = 0
    for i in range(8):                    # S-box sur chaque nibble
        y |= SBOX[(x >> (4 * i)) & 0xF] << (4 * i)
    return y ^ rotl(y, 5) ^ rotl(y, 11)   # diffusion

def key_schedule(key, rounds=ROUNDS):
    """key : 16 octets -> liste de sous-clés de 32 bits"""
    W = [int.from_bytes(key[4*i:4*i+4], 'big') for i in range(4)]
    k, sub = 0, []
    for i in range(rounds):
        k = rotl((k + W[i % 4] + C * (i + 1)) & M32, 7)
        sub.append(k)
    return sub

### Chiffrement / déchiffrement d'un bloc

In [2]:
def feistel_block(block, subkeys):
    L, R = block >> 32, block & M32
    for K in subkeys:
        L, R = R, L ^ F(R, K)
    return (R << 32) | L        # on échange les moitiés à la fin (comme DES)

def encrypt_block(block, subkeys):
    return feistel_block(block, subkeys)

def decrypt_block(block, subkeys):
    return feistel_block(block, subkeys[::-1])   # mêmes opérations, clés inversées

### Messages de taille quelconque : padding PKCS#7 + ECB

On complète le message avec $n$ octets valant $n$ ($1 \le n \le 8$) pour obtenir un multiple de 8 octets. Un message vide donne donc un bloc complet de padding.

In [3]:
def pad(data):
    n = 8 - len(data) % 8
    return data + bytes([n]) * n

def unpad(data):
    return data[:-data[-1]]

def encrypt(data, key):
    sk = key_schedule(key)
    data = pad(data)
    out = bytearray()
    for i in range(0, len(data), 8):
        b = int.from_bytes(data[i:i+8], 'big')
        out += encrypt_block(b, sk).to_bytes(8, 'big')
    return bytes(out)

def decrypt(data, key):
    sk = key_schedule(key)
    out = bytearray()
    for i in range(0, len(data), 8):
        b = int.from_bytes(data[i:i+8], 'big')
        out += decrypt_block(b, sk).to_bytes(8, 'big')
    return unpad(bytes(out))

In [4]:
key = os.urandom(16)
msg = b"Bonjour INSA Lyon !"
ct = encrypt(msg, key)
print("clé       :", key.hex())
print("clair     :", msg)
print("chiffré   :", ct.hex())
print("déchiffré :", decrypt(ct, key))

clé       : 95c74816f42a821c7f9e67d98d3debbc
clair     : b'Bonjour INSA Lyon !'
chiffré   : db05f550cc16b0fdcebc747f178ec3288144ffeb54861d38
déchiffré : b'Bonjour INSA Lyon !'


### Diffusion et confusion : effet avalanche

Pour vérifier la diffusion, on change **1 bit** du clair (ou de la clé) et on compte le nombre de bits du chiffré qui changent. Un bon chiffrement doit en changer environ la moitié, soit **32 bits sur 64**.

On compare avec un chiffrement trivial « XOR avec la clé » pour montrer la différence.

In [5]:
def hamming(a, b):
    return bin(a ^ b).count('1')

def avalanche(rounds, n=500):
    tot_pt, tot_key = 0, 0
    for _ in range(n):
        key = os.urandom(16)
        sk = key_schedule(key, rounds)
        p = random.getrandbits(64)
        bit = 1 << random.randrange(64)
        tot_pt += hamming(feistel_block(p, sk), feistel_block(p ^ bit, sk))
        # 1 bit de clé modifié
        k2 = bytearray(key); k2[random.randrange(16)] ^= 1 << random.randrange(8)
        tot_key += hamming(feistel_block(p, sk), feistel_block(p, key_schedule(bytes(k2), rounds)))
    return tot_pt / n, tot_key / n

print("tours | bits changés (1 bit clair) | bits changés (1 bit clé)")
for r in [1, 2, 3, 4, 6, 8, 16]:
    a, b = avalanche(r)
    print(f"{r:5d} | {a:26.1f} | {b:24.1f}")

# chiffrement trivial XOR
k = random.getrandbits(64); p = random.getrandbits(64)
print("\nXOR trivial : bits changés pour 1 bit de clair =", hamming(p ^ k, (p ^ 1) ^ k))

tours | bits changés (1 bit clair) | bits changés (1 bit clé)
    1 |                        5.1 |                      2.2
    2 |                       15.8 |                      7.1
    3 |                       26.3 |                     15.0
    4 |                       31.0 |                     23.6
    6 |                       31.9 |                     31.7
    8 |                       32.0 |                     31.9
   16 |                       32.0 |                     31.8

XOR trivial : bits changés pour 1 bit de clair = 1


**Interprétation :** avec 1 ou 2 tours, la modification ne touche qu'une moitié du bloc (propriété du Feistel : une moitié est recopiée telle quelle). À partir de 3–4 tours on atteint ~32 bits modifiés, c'est-à-dire un chiffré qui semble indépendant du précédent. Avec 16 tours on a donc une marge confortable.

Le XOR trivial, lui, ne change qu'un seul bit : aucune diffusion, et il est linéaire (un seul couple clair/chiffré donne la clé : $K = P \oplus C$). Notre design est donc nettement plus solide.

> Remarque : ce chiffrement n'a évidemment pas été analysé par la communauté (cryptanalyse linéaire/différentielle), il ne faut pas l'utiliser en pratique. De plus le mode ECB laisse fuiter les blocs répétés (voir test « motif répété » plus bas).

## 2. Vérification de la correction

On génère des clés aléatoires et on teste des messages de longueurs et de contenus variés, y compris des cas limites. Pour chaque message on vérifie que `decrypt(encrypt(m)) == m` avec 10 clés différentes.

In [6]:
tests = {
    "vide":                 b"",
    "1 octet":              b"A",
    "7 octets (impair)":    b"abcdefg",
    "8 octets (1 bloc)":    b"12345678",
    "9 octets":             b"123456789",
    "texte ASCII":          b"The quick brown fox jumps over the lazy dog",
    "UTF-8 (accents)":      "Chiffrement à clé symétrique, été, ça marche ?".encode(),
    "motif répété":         b"ABCDEFGH" * 8,
    "que des zéros":        bytes(32),
    "que des 0xFF":         b"\xff" * 33,
    "tous les octets":      bytes(range(256)),
    "binaire aléatoire":    os.urandom(1001),
    "gros message (100 ko)": os.urandom(100_000),
}

print(f"{'test':24s} {'taille':>7s} {'chiffré':>8s}  résultat")
for name, m in tests.items():
    ok = all(decrypt(encrypt(m, k), k) == m for k in [os.urandom(16) for _ in range(10)])
    c = encrypt(m, os.urandom(16))
    print(f"{name:24s} {len(m):7d} {len(c):8d}  {'OK' if ok else 'ERREUR'}")

test                      taille  chiffré  résultat
vide                           0        8  OK
1 octet                        1        8  OK
7 octets (impair)              7        8  OK
8 octets (1 bloc)              8       16  OK
9 octets                       9       16  OK
texte ASCII                   43       48  OK
UTF-8 (accents)               52       56  OK
motif répété                  64       72  OK
que des zéros                 32       40  OK
que des 0xFF                  33       40  OK
tous les octets              256      264  OK


binaire aléatoire           1001     1008  OK


gros message (100 ko)     100000   100008  OK


Tous les tests passent : le déchiffrement redonne exactement le message original, y compris pour le message vide (qui donne 1 bloc de padding) et les longueurs non multiples de 8.

**Remarque sur le motif répété** : en ECB, deux blocs de clair identiques donnent deux blocs chiffrés identiques :

In [7]:
c = encrypt(b"ABCDEFGH" * 4, os.urandom(16))
for i in range(0, len(c), 8):
    print(c[i:i+8].hex())

c5bf826eb51e199d
c5bf826eb51e199d
c5bf826eb51e199d
c5bf826eb51e199d
cd79a820bb3367a0


Les 4 premiers blocs sont identiques (le dernier est le bloc de padding). C'est une faiblesse du mode ECB, pas du chiffrement de bloc lui-même ; un mode comme CBC ou CTR résoudrait ce problème.